# 10 — MCP-Native Tool Architecture

Shows the low-level `BaseTool` / `ToolResult` API that underpins all tools in the framework.

Topics covered:
- Defining a tool by subclassing `BaseTool`
- Dual schema output: **MCP format** and **OpenAI function-calling format**
- Building a `ToolExecutionResultMessage` from a raw `ToolResult`
- Converting to both MCP and OpenAI wire formats

In [1]:
import json
from agent_framework.core.tools.base_tool import BaseTool, Tool, ToolResult
from agent_framework.core.messages.client_messages import ToolExecutionResultMessage

## Define a simple tool

In [2]:
class AddTool(BaseTool):
    """Add two numbers."""

    def __init__(self):
        self.tool_schema = Tool(
            name='add_numbers',
            description='Add two numbers together',
            inputSchema={
                'type': 'object',
                'properties': {
                    'a': {'type': 'number', 'description': 'First number'},
                    'b': {'type': 'number', 'description': 'Second number'},
                },
                'required': ['a', 'b'],
            },
        )

    async def execute(self, a: float, b: float) -> ToolResult:
        result = a + b
        return ToolResult(
            content=[{'type': 'text', 'text': json.dumps({'result': result, 'operation': f'{a} + {b}'})}],
            isError=False,
        )

    def get_schema(self) -> Tool:
        return self.tool_schema


tool = AddTool()
print('Tool defined:', tool.get_schema().name)

Tool defined: add_numbers


## Inspect both schema formats

In [3]:
print('MCP schema:')
print(json.dumps(tool.get_mcp_schema(), indent=2))

print('\nOpenAI schema:')
print(json.dumps(tool.get_openai_schema(), indent=2))

MCP schema:
{
  "name": "add_numbers",
  "description": "Add two numbers together",
  "inputSchema": {
    "type": "object",
    "properties": {
      "a": {
        "type": "number",
        "description": "First number"
      },
      "b": {
        "type": "number",
        "description": "Second number"
      }
    },
    "required": [
      "a",
      "b"
    ]
  }
}

OpenAI schema:
{
  "type": "function",
  "function": {
    "name": "add_numbers",
    "description": "Add two numbers together",
    "parameters": {
      "type": "object",
      "properties": {
        "a": {
          "type": "number",
          "description": "First number"
        },
        "b": {
          "type": "number",
          "description": "Second number"
        }
      },
      "required": [
        "a",
        "b"
      ]
    },
    "strict": true
  }
}


## Execute and build result messages

In [4]:
async def run():
    result = await tool.execute(a=5, b=7)
    print('ToolResult:', result.content)

    msg = ToolExecutionResultMessage.from_tool_result(
        tool_result=result,
        tool_call_id='call_abc123',
        tool_name='add_numbers',
    )
    print('\nOpenAI wire format:')
    print(json.dumps(msg.to_openai_format(), indent=2))

    print('\nMCP wire format:')
    print(json.dumps(msg.to_mcp_format(), indent=2))

await run()

ToolResult: [{'type': 'text', 'text': '{"result": 12, "operation": "5 + 7"}'}]

OpenAI wire format:
{
  "role": "tool",
  "tool_call_id": "call_abc123",
  "content": "{\"result\": 12, \"operation\": \"5 + 7\"}"
}

MCP wire format:
{
  "content": [
    {
      "type": "text",
      "text": "{\"result\": 12, \"operation\": \"5 + 7\"}"
    }
  ],
  "isError": false
}
